In [37]:
import pandas as pd
import os

# Define paths
sample_data_path = "/content/sample_data"
train_path = "/content/train.csv"
test_path = "/content/test.csv"

# List files in sample_data
print("Files in sample_data:")
print(os.listdir(sample_data_path))

# Read CSV files
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

import pandas as pd

train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")

Files in sample_data:
['README.md', 'anscombe.json', 'california_housing_train.csv', 'mnist_train_small.csv', 'california_housing_test.csv', 'mnist_test.csv']


In [39]:
print(train_df.head())

print(train_df.columns)

print(train_df.shape)

   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   
3   4  Select the most accurate option: What is Marti...   
4   5  Identify the correct statement: What is the co...   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   
2                                       Blueshifting   
3  Martin Heidegger believes that humans exist wi...   
4  Simultaneity is relative, meaning that two eve...   

                                                   B  \
0  Martin Heidegger believes that humans do not e...   
1  Accelerator-based light-ion fusion is a techni...   
2                                        Redshifting   
3  Martin Heidegger believes that humans do not e...   
4  Simultaneity is rel

In [40]:
print(test_df.head())

print(test_df.columns)

print(test_df.shape)

   id                                             prompt  \
0   1  Pick the best possible answer: What is the rel...   
1   2  What is the estimated redshift of CEERS-93316,...   
2   3  Pick the best possible answer: What is the rea...   
3   4  What is the significance of the redshift-dista...   
4   5  What is the Landau-Lifshitz-Gilbert equation u...   

                                                   A  \
0  For every eigenstate of one Hamiltonian, its p...   
1  Approximately z = 6.0, corresponding to 1 bill...   
2  The sun appears yellowish due to a reflection ...   
3  Observations of the redshift-distance relation...   
4  The Landau-Lifshitz-Gilbert equation is a diff...   

                                                   B  \
0  For every eigenstate of one Hamiltonian, its p...   
1  Approximately z = 16.7, corresponding to 235.8...   
2  The longer wavelengths of light, such as red a...   
3  Observations of the redshift-distance relation...   
4  The Landau-Lifshitz

In [41]:
!pip install -q transformers datasets accelerate evaluate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00


In [42]:
import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split

In [43]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

cuda
Tesla T4


In [44]:
train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")

print(train_df.shape)
print(test_df.shape)

train_df.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [45]:
label2id = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

id2label = {
    0:"A",
    1:"B",
    2:"C",
    3:"D",
    4:"E"
}

train_df["label"] = train_df["answer"].map(label2id)

train_df[["answer","label"]].head()

,answer,label
0,B,1
1,A,0
2,C,2
3,B,1
4,A,0


In [46]:
train_data, valid_data = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["label"]
)

print(train_data.shape)
print(valid_data.shape)

(1800, 9)
(200, 9)


In [47]:
train_dataset = Dataset.from_pandas(train_data.reset_index(drop=True))
valid_dataset = Dataset.from_pandas(valid_data.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df)

In [48]:
MODEL_NAME = "microsoft/deberta-v3-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [49]:
def preprocess(example):

    text = (
        "Question: " + example["prompt"] +
        "\nA. " + example["A"] +
        "\nB. " + example["B"] +
        "\nC. " + example["C"] +
        "\nD. " + example["D"] +
        "\nE. " + example["E"]
    )

    tokenized = tokenizer(
        text,
        truncation=True,
        padding=False,
        max_length=512
    )

    tokenized["labels"] = example["label"]

    return tokenized

In [50]:
train_dataset = train_dataset.map(preprocess)
valid_dataset = valid_dataset.map(preprocess)

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [51]:
columns_to_remove = [
    "id",
    "prompt",
    "A",
    "B",
    "C",
    "D",
    "E",
    "answer",
    "label"
]

train_dataset = train_dataset.remove_columns(columns_to_remove)
valid_dataset = valid_dataset.remove_columns(columns_to_remove)

In [52]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [53]:
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "microsoft/deberta-v3-small"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5,
    id2label=id2label,
    label2id=label2id
)

model.to(device)

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-5): 6 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNo

In [56]:
import numpy as np
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions)
    }

In [57]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./deberta_output",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=50,

    load_best_model_at_end=True,

    metric_for_best_model="accuracy",

    greater_is_better=True,

    report_to="none",

    fp16=torch.cuda.is_available(),

    save_total_limit=1
)

In [58]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [59]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.251700,0.928584,0.675000
2,0.113800,0.043513,0.995000
3,0.010200,0.005182,1.000000


TrainOutput(global_step=675, training_loss=0.6085568007716426, metrics={'train_runtime': 194.9902, 'train_samples_per_second': 27.694, 'train_steps_per_second': 3.462, 'total_flos': 491249449637760.0, 'train_loss': 0.6085568007716426, 'epoch': 3.0})

In [60]:
results = trainer.evaluate()

print(results)

{'eval_loss': 0.005181851331144571, 'eval_accuracy': 1.0, 'eval_runtime': 1.6218, 'eval_samples_per_second': 123.319, 'eval_steps_per_second': 8.016, 'epoch': 3.0}


In [61]:
trainer.save_model("deberta_checkpoint")
tokenizer.save_pretrained("deberta_checkpoint")

('deberta_checkpoint/tokenizer_config.json',
 'deberta_checkpoint/special_tokens_map.json',
 'deberta_checkpoint/spm.model',
 'deberta_checkpoint/added_tokens.json',
 'deberta_checkpoint/tokenizer.json')

In [62]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("deberta_checkpoint")

model = AutoModelForSequenceClassification.from_pretrained(
    "deberta_checkpoint"
).to(device)

model.eval()

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-5): 6 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNo

In [63]:
sample = valid_data.iloc[0]

text = (
    "Question: " + sample["prompt"] +
    "\nA. " + sample["A"] +
    "\nB. " + sample["B"] +
    "\nC. " + sample["C"] +
    "\nD. " + sample["D"] +
    "\nE. " + sample["E"]
)

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    logits = model(**inputs).logits

prediction = torch.argmax(logits, dim=-1).item()

print("Predicted :", id2label[prediction])
print("Actual    :", sample["answer"])

Predicted : A
Actual    : A


In [64]:
import transformers
print(transformers.__version__)

4.51.3


# **Part 3 — Fine-tune RoBERTa-base**

In [65]:
from transformers import AutoTokenizer

ROBERTA_MODEL = "roberta-base"

roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [66]:
def roberta_preprocess(example):

    text = (
        "Question: " + example["prompt"] +
        "\nA. " + example["A"] +
        "\nB. " + example["B"] +
        "\nC. " + example["C"] +
        "\nD. " + example["D"] +
        "\nE. " + example["E"]
    )

    tokenized = roberta_tokenizer(
        text,
        truncation=True,
        padding=False,
        max_length=512
    )

    tokenized["labels"] = example["label"]

    return tokenized

In [67]:
from datasets import Dataset

train_dataset_roberta = Dataset.from_pandas(
    train_data.reset_index(drop=True)
)

valid_dataset_roberta = Dataset.from_pandas(
    valid_data.reset_index(drop=True)
)

In [68]:
train_dataset_roberta = train_dataset_roberta.map(
    roberta_preprocess
)

valid_dataset_roberta = valid_dataset_roberta.map(
    roberta_preprocess
)

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [69]:
columns_to_remove = [
    "id",
    "prompt",
    "A",
    "B",
    "C",
    "D",
    "E",
    "answer",
    "label"
]

train_dataset_roberta = train_dataset_roberta.remove_columns(columns_to_remove)

valid_dataset_roberta = valid_dataset_roberta.remove_columns(columns_to_remove)

In [70]:
from transformers import DataCollatorWithPadding

roberta_data_collator = DataCollatorWithPadding(
    tokenizer=roberta_tokenizer
)

In [71]:
from transformers import AutoModelForSequenceClassification

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    ROBERTA_MODEL,
    num_labels=5,
    id2label=id2label,
    label2id=label2id
)

roberta_model.to(device)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [72]:
from transformers import TrainingArguments

roberta_training_args = TrainingArguments(
    output_dir="./roberta_output",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=50,

    load_best_model_at_end=True,

    metric_for_best_model="accuracy",

    greater_is_better=True,

    report_to="none",

    fp16=torch.cuda.is_available(),

    save_total_limit=1
)

In [73]:
from transformers import Trainer

roberta_trainer = Trainer(
    model=roberta_model,
    args=roberta_training_args,
    train_dataset=train_dataset_roberta,
    eval_dataset=valid_dataset_roberta,
    processing_class=roberta_tokenizer,
    data_collator=roberta_data_collator,
    compute_metrics=compute_metrics,
)

In [74]:
roberta_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.791700,0.248801,0.955000
2,0.005100,0.002681,1.000000
3,0.002900,0.001769,1.000000


TrainOutput(global_step=675, training_loss=0.436712092117027, metrics={'train_runtime': 327.0707, 'train_samples_per_second': 16.51, 'train_steps_per_second': 2.064, 'total_flos': 1038395749234176.0, 'train_loss': 0.436712092117027, 'epoch': 3.0})

In [75]:
results = roberta_trainer.evaluate()

print(results)

{'eval_loss': 0.002680940553545952, 'eval_accuracy': 1.0, 'eval_runtime': 1.2264, 'eval_samples_per_second': 163.08, 'eval_steps_per_second': 10.6, 'epoch': 3.0}


In [76]:
roberta_trainer.save_model("roberta_checkpoint")

roberta_tokenizer.save_pretrained(
    "roberta_checkpoint"
)

('roberta_checkpoint/tokenizer_config.json',
 'roberta_checkpoint/special_tokens_map.json',
 'roberta_checkpoint/vocab.json',
 'roberta_checkpoint/merges.txt',
 'roberta_checkpoint/added_tokens.json',
 'roberta_checkpoint/tokenizer.json')

In [77]:
sample = valid_data.iloc[0]

text = (
    "Question: " + sample["prompt"] +
    "\nA. " + sample["A"] +
    "\nB. " + sample["B"] +
    "\nC. " + sample["C"] +
    "\nD. " + sample["D"] +
    "\nE. " + sample["E"]
)

inputs = roberta_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    logits = roberta_model(**inputs).logits

prediction = torch.argmax(logits, dim=-1).item()

print("Predicted :", id2label[prediction])
print("Actual    :", sample["answer"])

Predicted : A
Actual    : A


# **Part 4 — Ensemble Inference**

In [78]:
import torch
import numpy as np
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# DeBERTa
deberta_tokenizer = AutoTokenizer.from_pretrained("deberta_checkpoint")

deberta_model = AutoModelForSequenceClassification.from_pretrained(
    "deberta_checkpoint"
).to(device)

deberta_model.eval()

# RoBERTa
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta_checkpoint")

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    "roberta_checkpoint"
).to(device)

roberta_model.eval()

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [79]:
test_df = pd.read_csv("/content/test.csv")

test_df.head()

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [80]:
def predict_probabilities(
    model,
    tokenizer,
    dataframe,
    batch_size=16
):

    all_probs = []

    model.eval()

    with torch.inference_mode():

        for i in range(0, len(dataframe), batch_size):

            batch = dataframe.iloc[i:i+batch_size]

            texts = []

            for _, row in batch.iterrows():

                text = (
                    "Question: " + row["prompt"] +
                    "\nA. " + row["A"] +
                    "\nB. " + row["B"] +
                    "\nC. " + row["C"] +
                    "\nD. " + row["D"] +
                    "\nE. " + row["E"]
                )

                texts.append(text)

            inputs = tokenizer(
                texts,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            )

            inputs = {
                k: v.to(device)
                for k, v in inputs.items()
            }

            logits = model(**inputs).logits

            probs = torch.softmax(
                logits,
                dim=-1
            )

            all_probs.append(
                probs.cpu().numpy()
            )

    return np.concatenate(all_probs)

In [81]:
deberta_probs = predict_probabilities(
    deberta_model,
    deberta_tokenizer,
    test_df
)

deberta_probs.shape

(500, 5)

In [84]:
roberta_probs = predict_probabilities(
    roberta_model,
    roberta_tokenizer,
    test_df
)

roberta_probs.shape

(500, 5)

In [85]:
ensemble_probs = (
    deberta_probs +
    roberta_probs
) / 2

In [86]:
top3 = np.argsort(
    -ensemble_probs,
    axis=1
)[:, :3]

top3[:5]

array([[0, 3, 4],
       [1, 3, 2],
       [1, 3, 4],
       [4, 0, 2],
       [2, 3, 0]])

In [87]:
id2label = {
    0:"A",
    1:"B",
    2:"C",
    3:"D",
    4:"E"
}

predictions = []

for row in top3:

    pred = " ".join(
        id2label[i]
        for i in row
    )

    predictions.append(pred)

predictions[:5]

['A D E', 'B D C', 'B D E', 'E A C', 'C D A']

In [88]:
submission = pd.DataFrame({

    "id": test_df["id"],

    "prediction": predictions

})

submission.head()

,id,prediction
0,1,A D E
1,2,B D C
2,3,B D E
3,4,E A C
4,5,C D A


In [89]:
submission.to_csv(
    "submission.csv",
    index=False
)

print("submission.csv created successfully!")

submission.csv created successfully!


In [90]:
submission.head(10)

,id,prediction
0,1,A D E
1,2,B D C
2,3,B D E
3,4,E A C
4,5,C D A
5,6,D C A
6,7,E A B
7,8,B D E
8,9,C D A
9,10,B D E


# **Next: Test-Time Augmentation (TTA)**

In [91]:
def build_text(row, version=0):

    if version == 0:
        return (
            "Question: " + row["prompt"] +
            "\nA. " + row["A"] +
            "\nB. " + row["B"] +
            "\nC. " + row["C"] +
            "\nD. " + row["D"] +
            "\nE. " + row["E"]
        )

    elif version == 1:
        return (
            "Choose the best answer.\n\n"
            + row["prompt"] +
            "\n\nOptions:\n"
            "A. " + row["A"] +
            "\nB. " + row["B"] +
            "\nC. " + row["C"] +
            "\nD. " + row["D"] +
            "\nE. " + row["E"]
        )

    elif version == 2:
        return (
            row["prompt"] +
            "\n\nSelect one option.\n\n"
            "A. " + row["A"] +
            "\nB. " + row["B"] +
            "\nC. " + row["C"] +
            "\nD. " + row["D"] +
            "\nE. " + row["E"]
        )

In [92]:
def predict_tta(model, tokenizer, dataframe, version, batch_size=16):

    model.eval()

    probabilities = []

    with torch.inference_mode():

        for start in range(0, len(dataframe), batch_size):

            batch = dataframe.iloc[start:start+batch_size]

            texts = [
                build_text(row, version)
                for _, row in batch.iterrows()
            ]

            inputs = tokenizer(
                texts,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            )

            inputs = {
                k: v.to(device)
                for k, v in inputs.items()
            }

            logits = model(**inputs).logits

            probs = torch.softmax(
                logits,
                dim=-1
            )

            probabilities.append(
                probs.cpu().numpy()
            )

    return np.concatenate(probabilities)

In [93]:
deberta_v1 = predict_tta(
    deberta_model,
    deberta_tokenizer,
    test_df,
    version=0
)

deberta_v2 = predict_tta(
    deberta_model,
    deberta_tokenizer,
    test_df,
    version=1
)

deberta_v3 = predict_tta(
    deberta_model,
    deberta_tokenizer,
    test_df,
    version=2
)

deberta_probs_tta = (
    deberta_v1 +
    deberta_v2 +
    deberta_v3
) / 3

In [94]:
roberta_v1 = predict_tta(
    roberta_model,
    roberta_tokenizer,
    test_df,
    version=0
)

roberta_v2 = predict_tta(
    roberta_model,
    roberta_tokenizer,
    test_df,
    version=1
)

roberta_v3 = predict_tta(
    roberta_model,
    roberta_tokenizer,
    test_df,
    version=2
)

roberta_probs_tta = (
    roberta_v1 +
    roberta_v2 +
    roberta_v3
) / 3

In [97]:
final_probs = (
    deberta_probs_tta +
    roberta_probs_tta
) / 2

In [96]:
# final_probs = (
#     0.6 * deberta_probs_tta +
#     0.4 * roberta_probs_tta
# )

In [98]:
top3 = np.argsort(
    -final_probs,
    axis=1
)[:, :3]

predictions = []

for row in top3:

    predictions.append(
        " ".join(
            id2label[x]
            for x in row
        )
    )

In [99]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "prediction": predictions
})

submission.to_csv(
    "submission_tta.csv",
    index=False
)

submission.head()

,id,prediction
0,1,A D E
1,2,B D C
2,3,B D E
3,4,E A C
4,5,C D A


# **MILESTONE 5 STARTS HERE**

# QN1

In [100]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load fine-tuned DeBERTa checkpoint
deberta_tokenizer = AutoTokenizer.from_pretrained("deberta_checkpoint")

deberta_model = AutoModelForSequenceClassification.from_pretrained(
    "deberta_checkpoint"
).to(device)

deberta_model.eval()

# Load test data
import pandas as pd
test_df = pd.read_csv("/content/test.csv")

# Select row index 25
row = test_df.iloc[25]

# Prepare input
text = (
    "Question: " + row["prompt"] +
    "\nA. " + row["A"] +
    "\nB. " + row["B"] +
    "\nC. " + row["C"] +
    "\nD. " + row["D"] +
    "\nE. " + row["E"]
)

inputs = deberta_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {k: v.to(device) for k, v in inputs.items()}

# Inference
with torch.inference_mode():
    outputs = deberta_model(**inputs)
    logits = outputs.logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

# Label mapping
id2label = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

# Print probabilities
print("Class Probabilities:")
for i, p in enumerate(probs):
    print(f"{id2label[i]} : {p:.6f}")

# Highest probability
pred_idx = probs.argmax()

print("\nHighest Probability Option:", id2label[pred_idx])
print("Probability:", probs[pred_idx])

Class Probabilities:
A : 0.001713
B : 0.001142
C : 0.000625
D : 0.000257
E : 0.996264

Highest Probability Option: E
Probability: 0.9962637


# QN2

In [101]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load test data
test_df = pd.read_csv("/content/test.csv")

# Load DeBERTa
deberta_tokenizer = AutoTokenizer.from_pretrained("deberta_checkpoint")
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    "deberta_checkpoint"
).to(device)
deberta_model.eval()

# Load RoBERTa
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta_checkpoint")
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    "roberta_checkpoint"
).to(device)
roberta_model.eval()

# Select row index 25
row = test_df.iloc[25]

# Create input text
text = (
    "Question: " + row["prompt"] +
    "\nA. " + row["A"] +
    "\nB. " + row["B"] +
    "\nC. " + row["C"] +
    "\nD. " + row["D"] +
    "\nE. " + row["E"]
)

# -------- DeBERTa --------
inputs = deberta_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.inference_mode():
    logits = deberta_model(**inputs).logits
    deberta_probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

# -------- RoBERTa --------
inputs = roberta_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.inference_mode():
    logits = roberta_model(**inputs).logits
    roberta_probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

# -------- Average Probabilities --------
avg_probs = (deberta_probs + roberta_probs) / 2

id2label = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

print("DeBERTa Probabilities")
for i, p in enumerate(deberta_probs):
    print(f"{id2label[i]} : {p:.6f}")

print("\nRoBERTa Probabilities")
for i, p in enumerate(roberta_probs):
    print(f"{id2label[i]} : {p:.6f}")

print("\nAverage Probabilities")
for i, p in enumerate(avg_probs):
    print(f"{id2label[i]} : {p:.6f}")

pred_idx = np.argmax(avg_probs)

print("\nHighest Averaged Probability Option:", id2label[pred_idx])
print("Probability:", avg_probs[pred_idx])

DeBERTa Probabilities
A : 0.001713
B : 0.001142
C : 0.000625
D : 0.000257
E : 0.996264

RoBERTa Probabilities
A : 0.000718
B : 0.000629
C : 0.001043
D : 0.000665
E : 0.996946

Average Probabilities
A : 0.001215
B : 0.000885
C : 0.000834
D : 0.000461
E : 0.996605

Highest Averaged Probability Option: E
Probability: 0.9966049


# QN3

In [102]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load test data
test_df = pd.read_csv("/content/test.csv")

# Load DeBERTa
deberta_tokenizer = AutoTokenizer.from_pretrained("deberta_checkpoint")
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    "deberta_checkpoint"
).to(device)
deberta_model.eval()

# Load RoBERTa
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta_checkpoint")
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    "roberta_checkpoint"
).to(device)
roberta_model.eval()

# Select row index 25
row = test_df.iloc[25]

# Build input text
text = (
    "Question: " + row["prompt"] +
    "\nA. " + row["A"] +
    "\nB. " + row["B"] +
    "\nC. " + row["C"] +
    "\nD. " + row["D"] +
    "\nE. " + row["E"]
)

# ---------------- DeBERTa ----------------
inputs = deberta_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.inference_mode():
    logits = deberta_model(**inputs).logits
    deberta_probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

# ---------------- RoBERTa ----------------
inputs = roberta_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.inference_mode():
    logits = roberta_model(**inputs).logits
    roberta_probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

# ---------------- Weighted Ensemble ----------------
final_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)

id2label = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

print("Weighted Ensemble Probabilities\n")

for i, p in enumerate(final_probs):
    print(f"{id2label[i]} : {p:.6f}")

best_idx = np.argmax(final_probs)

print("\nRank 1 Option :", id2label[best_idx])
print("Probability   :", final_probs[best_idx])

Weighted Ensemble Probabilities

A : 0.001414
B : 0.000988
C : 0.000750
D : 0.000379
E : 0.996468

Rank 1 Option : E
Probability   : 0.9964684


# QN4

In [103]:
import numpy as np

# Label mapping
id2label = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

# Get indices sorted by descending probability
top3_indices = np.argsort(final_probs)[::-1][:3]

# Convert indices to answer options
top3_prediction = " ".join(id2label[idx] for idx in top3_indices)

# Display all rankings
print("Ranked Options:")
for rank, idx in enumerate(np.argsort(final_probs)[::-1], start=1):
    print(f"{rank}. {id2label[idx]} ({final_probs[idx]:.6f})")

print("\nTop-3 Prediction String:")
print(top3_prediction)

Ranked Options:
1. E (0.996468)
2. A (0.001414)
3. B (0.000988)
4. C (0.000750)
5. D (0.000379)

Top-3 Prediction String:
E A B


# QN5

In [104]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load test data
test_df = pd.read_csv("/content/test.csv")

# Label mapping
id2label = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

# Load DeBERTa
deberta_tokenizer = AutoTokenizer.from_pretrained("deberta_checkpoint")
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    "deberta_checkpoint"
).to(device)
deberta_model.eval()

# Load RoBERTa
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta_checkpoint")
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    "roberta_checkpoint"
).to(device)
roberta_model.eval()

predictions = []

# Inference on every test sample
for _, row in test_df.iterrows():

    text = (
        "Question: " + row["prompt"] +
        "\nA. " + row["A"] +
        "\nB. " + row["B"] +
        "\nC. " + row["C"] +
        "\nD. " + row["D"] +
        "\nE. " + row["E"]
    )

    # -------- DeBERTa --------
    inputs = deberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        logits = deberta_model(**inputs).logits
        deberta_probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    # -------- RoBERTa --------
    inputs = roberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        logits = roberta_model(**inputs).logits
        roberta_probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    # -------- Weighted Ensemble --------
    final_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)

    # Top-3 predictions
    top3 = np.argsort(final_probs)[::-1][:3]

    prediction = " ".join(id2label[idx] for idx in top3)

    predictions.append(prediction)

# Create submission
submission = pd.DataFrame({
    "id": test_df["id"],
    "prediction": predictions
})

# Save file
submission.to_csv("submission.csv", index=False)

print(submission.head())

print("\nNumber of prediction rows (excluding header):", len(submission))

   id prediction
0   1      A D E
1   2      B D C
2   3      B E D
3   4      E A C
4   5      C D A

Number of prediction rows (excluding header): 500


# QN6

In [105]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load fine-tuned DeBERTa
tokenizer = AutoTokenizer.from_pretrained("deberta_checkpoint")
model = AutoModelForSequenceClassification.from_pretrained(
    "deberta_checkpoint"
).to(device)

model.eval()

# Load test data
test_df = pd.read_csv("/content/test.csv")

# Label mapping
id2label = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

changed_predictions = 0

for idx in range(50):

    row = test_df.iloc[idx]

    # ---------------- Original Prompt ----------------
    original_text = (
        "Question: " + row["prompt"] +
        "\nA. " + row["A"] +
        "\nB. " + row["B"] +
        "\nC. " + row["C"] +
        "\nD. " + row["D"] +
        "\nE. " + row["E"]
    )

    # ---------------- Augmented Prompt ----------------
    augmented_text = (
        "Answer the following multiple-choice question carefully:\n\n"
        + "Question: " + row["prompt"] +
        "\nA. " + row["A"] +
        "\nB. " + row["B"] +
        "\nC. " + row["C"] +
        "\nD. " + row["D"] +
        "\nE. " + row["E"]
    )

    # ---------- Original Prediction ----------
    inputs = tokenizer(
        original_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        logits = model(**inputs).logits
        probs_original = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    original_top1 = np.argmax(probs_original)

    # ---------- Augmented Prediction ----------
    inputs = tokenizer(
        augmented_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        logits = model(**inputs).logits
        probs_augmented = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    # ---------- TTA Average ----------
    probs_tta = (probs_original + probs_augmented) / 2

    tta_top1 = np.argmax(probs_tta)

    if original_top1 != tta_top1:
        changed_predictions += 1

print("Number of changed Top-1 predictions:", changed_predictions)

Number of changed Top-1 predictions: 0


# QN7

In [106]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load test data
test_df = pd.read_csv("/content/test.csv")

# Load DeBERTa
deberta_tokenizer = AutoTokenizer.from_pretrained("deberta_checkpoint")
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    "deberta_checkpoint"
).to(device)
deberta_model.eval()

# Load RoBERTa
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta_checkpoint")
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    "roberta_checkpoint"
).to(device)
roberta_model.eval()

id2label = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

different_predictions = 0

for idx in range(100):

    row = test_df.iloc[idx]

    text = (
        "Question: " + row["prompt"] +
        "\nA. " + row["A"] +
        "\nB. " + row["B"] +
        "\nC. " + row["C"] +
        "\nD. " + row["D"] +
        "\nE. " + row["E"]
    )

    # ---------- DeBERTa ----------
    inputs = deberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        logits = deberta_model(**inputs).logits
        deberta_probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    deberta_top1 = np.argmax(deberta_probs)

    # ---------- RoBERTa ----------
    inputs = roberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        logits = roberta_model(**inputs).logits
        roberta_probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    # ---------- Weighted Ensemble ----------
    ensemble_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)

    ensemble_top1 = np.argmax(ensemble_probs)

    if deberta_top1 != ensemble_top1:
        different_predictions += 1

print("Number of rows with different Top-1 predictions:", different_predictions)

Number of rows with different Top-1 predictions: 0


# QN8

In [107]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load test data
test_df = pd.read_csv("/content/test.csv")

# Load DeBERTa
deberta_tokenizer = AutoTokenizer.from_pretrained("deberta_checkpoint")
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    "deberta_checkpoint"
).to(device)
deberta_model.eval()

# Load RoBERTa
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta_checkpoint")
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    "roberta_checkpoint"
).to(device)
roberta_model.eval()

positive_gain = 0

for idx in range(100):

    row = test_df.iloc[idx]

    text = (
        "Question: " + row["prompt"] +
        "\nA. " + row["A"] +
        "\nB. " + row["B"] +
        "\nC. " + row["C"] +
        "\nD. " + row["D"] +
        "\nE. " + row["E"]
    )

    # ---------- DeBERTa ----------
    inputs = deberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        logits = deberta_model(**inputs).logits
        deberta_probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    # ---------- RoBERTa ----------
    inputs = roberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        logits = roberta_model(**inputs).logits
        roberta_probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    # ---------- Weighted Ensemble ----------
    ensemble_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)

    # Highest confidence
    deberta_confidence = np.max(deberta_probs)
    ensemble_confidence = np.max(ensemble_probs)

    confidence_gain = ensemble_confidence - deberta_confidence

    if confidence_gain > 0:
        positive_gain += 1

print("Rows with positive confidence gain:", positive_gain)

Rows with positive confidence gain: 99


# QN9

In [108]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load test data
test_df = pd.read_csv("/content/test.csv")

# Load DeBERTa
deberta_tokenizer = AutoTokenizer.from_pretrained("deberta_checkpoint")
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    "deberta_checkpoint"
).to(device)
deberta_model.eval()

# Load RoBERTa
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta_checkpoint")
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    "roberta_checkpoint"
).to(device)
roberta_model.eval()

id2label = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

changed_top3 = 0

for idx in range(100):

    row = test_df.iloc[idx]

    text = (
        f"Question: {row['prompt']}"
        f"\nA. {row['A']}"
        f"\nB. {row['B']}"
        f"\nC. {row['C']}"
        f"\nD. {row['D']}"
        f"\nE. {row['E']}"
    )

    # ---------- DeBERTa ----------
    inputs = deberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        logits = deberta_model(**inputs).logits
        deberta_probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    # ---------- RoBERTa ----------
    inputs = roberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        logits = roberta_model(**inputs).logits
        roberta_probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    # ---------- Weighted Ensemble ----------
    ensemble_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)

    # Top-3 rankings
    deberta_top3 = [id2label[i] for i in np.argsort(deberta_probs)[::-1][:3]]
    ensemble_top3 = [id2label[i] for i in np.argsort(ensemble_probs)[::-1][:3]]

    # Compare ordered Top-3
    if deberta_top3 != ensemble_top3:
        changed_top3 += 1

print("Rows with changed ordered Top-3 rankings:", changed_top3)

Rows with changed ordered Top-3 rankings: 9


# QN10

In [110]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load fine-tuned models
deberta_tokenizer = AutoTokenizer.from_pretrained("deberta_checkpoint")
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    "deberta_checkpoint"
).to(device)
deberta_model.eval()

roberta_tokenizer = AutoTokenizer.from_pretrained("roberta_checkpoint")
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    "roberta_checkpoint"
).to(device)
roberta_model.eval()

# Use the first 100 validation samples
df = valid_data.iloc[:100].reset_index(drop=True)

id2label = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

# MAP@3 function
def apk(actual, predicted, k=3):
    predicted = predicted[:k]
    for i, p in enumerate(predicted):
        if p == actual:
            return 1.0 / (i + 1)
    return 0.0

scores = []

for _, row in df.iterrows():

    text = (
        f"Question: {row['prompt']}"
        f"\nA. {row['A']}"
        f"\nB. {row['B']}"
        f"\nC. {row['C']}"
        f"\nD. {row['D']}"
        f"\nE. {row['E']}"
    )

    # -------- DeBERTa --------
    inputs = deberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        deberta_probs = torch.softmax(
            deberta_model(**inputs).logits,
            dim=-1
        ).cpu().numpy()[0]

    # -------- RoBERTa --------
    inputs = roberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        roberta_probs = torch.softmax(
            roberta_model(**inputs).logits,
            dim=-1
        ).cpu().numpy()[0]

    # -------- Weighted Ensemble --------
    final_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)

    # Top-3 predictions
    top3 = np.argsort(final_probs)[::-1][:3]
    top3_labels = [id2label[i] for i in top3]

    # Ground truth (A/B/C/D/E)
    actual = row["answer"]

    scores.append(apk(actual, top3_labels))

# Final MAP@3
map3 = np.mean(scores)

print(f"MAP@3 = {map3:.4f}")

MAP@3 = 1.0000
